# 10B - All CV Model Confidence Audit

Purpose: compute high-confidence errors for the trained computer-vision models on the grouped test split.

Definition used here matches notebooks 07 and 09:

```text
high_confidence_error = prediction is wrong AND max softmax confidence >= 0.95
```

This notebook does not train models. It loads saved `.keras` artifacts and evaluates them on `data/splits_grouped/test.csv`.

## Important Split-Compatibility Warning

This notebook evaluates saved model artifacts on `data/splits_grouped/test.csv`. That audit is valid for models trained with the grouped split, such as the rerun EfficientNetB0/ResNet50/XAI-corrected experiments.

Older baseline CNN and MobileNetV2 artifacts were originally trained/evaluated against the earlier split (`data/splits`) whose leakage audit reported overlapping source images. Therefore, their high-confidence error counts from this grouped-test audit are diagnostic only and must not be used as final model-selection evidence unless those models are retrained on the grouped split.

For the report table, use stored notebook/classification-report metrics for old baseline/MobileNet experiments, and use high-confidence error counts only where split compatibility is clear.

## Setup

If running from VS Code/Jupyter, the cell below finds the repository root and adds it to `sys.path` so `src.*` imports work.

In [ ]:
from __future__ import annotations

import gc
import json
import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
from IPython.display import display
from sklearn.metrics import classification_report

ROOT = Path.cwd().resolve()
while not (ROOT / "src").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import IMAGE_SIZE
from src.models.resnet50_model import ResNet50Preprocess

print("Repository root:", ROOT)
print("TensorFlow:", tf.__version__)
print("Image size:", IMAGE_SIZE)

## Model Registry

The registry below follows the models used in the report table. If any artifact is missing locally, the audit will skip it and record the missing path.

In [ ]:
MODEL_SPECS = [
    {
        "model_label": "Baseline CNN",
        "experiment_slug": "baseline_cnn",
        "model_path": ROOT / "models" / "baseline_cnn_best.keras",
    },
    {
        "model_label": "Baseline CNN + augmentation",
        "experiment_slug": "baseline_cnn_aug_only",
        "model_path": ROOT / "models" / "experiments" / "baseline_cnn_aug_only.keras",
    },
    {
        "model_label": "Baseline CNN + aug + oversample",
        "experiment_slug": "baseline_cnn_aug_oversampled",
        "model_path": ROOT / "models" / "experiments" / "baseline_cnn_aug_oversampled.keras",
    },
    {
        "model_label": "MobileNetV2 head",
        "experiment_slug": "mobilenetv2_aug_oversampled",
        "model_path": ROOT / "models" / "experiments" / "mobilenetv2_aug_oversampled.keras",
    },
    {
        "model_label": "MobileNetV2 fine-tuned",
        "experiment_slug": "mobilenetv2_aug_oversampled_finetuned_wsl",
        "model_path": ROOT / "models" / "experiments" / "mobilenetv2_aug_oversampled_finetuned_wsl.keras",
    },
    {
        "model_label": "EfficientNetB0 head",
        "experiment_slug": "efficientnetb0_aug_oversampled",
        "model_path": ROOT / "models" / "experiments" / "efficientnetb0_aug_oversampled.keras",
    },
    {
        "model_label": "EfficientNetB0 fine-tuned",
        "experiment_slug": "efficientnetb0_aug_oversampled_finetuned_wsl",
        "model_path": ROOT / "models" / "experiments" / "efficientnetb0_aug_oversampled_finetuned_wsl.keras",
    },
    {
        "model_label": "EfficientNetB0 XAI-corrected",
        "experiment_slug": "efficientnetb0_xai_corrected",
        "model_path": ROOT / "models" / "experiments" / "efficientnetb0_xai_corrected.keras",
    },
    {
        "model_label": "ResNet50 head",
        "experiment_slug": "resnet50_aug_oversampled",
        "model_path": ROOT / "models" / "experiments" / "resnet50_aug_oversampled.keras",
    },
    {
        "model_label": "ResNet50 fine-tuned",
        "experiment_slug": "resnet50_aug_oversampled_finetuned_wsl",
        "model_path": ROOT / "models" / "experiments" / "resnet50_aug_oversampled_finetuned_wsl.keras",
    },
]

model_registry_df = pd.DataFrame(
    [{**spec, "model_path": str(spec["model_path"]), "exists": spec["model_path"].exists()} for spec in MODEL_SPECS]
)
display(model_registry_df)

## Load Test Split

The helper below converts Windows `D:\\...` paths to `/mnt/d/...` when running under WSL/Linux, and does the reverse when needed on Windows.

In [ ]:
CLASS_NAMES_PATH = ROOT / "models" / "class_names.json"
TEST_SPLIT_PATH = ROOT / "data" / "splits_grouped" / "test.csv"
OUTPUT_DIR = ROOT / "outputs" / "final_evaluation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(CLASS_NAMES_PATH, "r", encoding="utf-8-sig") as f:
    class_names = json.load(f)

test_df = pd.read_csv(TEST_SPLIT_PATH)
test_df["class_index"] = test_df["class_index"].astype(int)

def normalise_image_path(path_value: str) -> str:
    raw = str(path_value)
    candidate = Path(raw)
    if candidate.exists():
        return str(candidate)

    # Windows path in WSL/Linux, e.g. d:\\folder\\image.jpg -> /mnt/d/folder/image.jpg
    if len(raw) >= 3 and raw[1:3] == ":\\" and os.name != "nt":
        drive = raw[0].lower()
        rest = raw[3:].replace("\\", "/")
        converted = Path(f"/mnt/{drive}/{rest}")
        if converted.exists():
            return str(converted)

    # WSL path in Windows, e.g. /mnt/d/folder/image.jpg -> D:\\folder\\image.jpg
    if raw.startswith("/mnt/") and os.name == "nt":
        parts = raw.split("/")
        if len(parts) > 3:
            drive = parts[2].upper() + ":"
            converted = Path(drive + "\\" + "\\".join(parts[3:]))
            if converted.exists():
                return str(converted)

    return raw

test_df["resolved_image_path"] = test_df["image_path"].map(normalise_image_path)
missing_images = test_df[~test_df["resolved_image_path"].map(lambda p: Path(p).exists())]
print("Test rows:", len(test_df))
print("Missing resolved images:", len(missing_images))
if len(missing_images):
    display(missing_images.head())
    raise FileNotFoundError("Some test images could not be resolved. Check WSL/Windows paths.")

display(test_df.head())

## Audit Helpers

In [ ]:
BATCH_SIZE = 32
HIGH_CONFIDENCE_THRESHOLD = 0.95

def decode_resize_image(path: tf.Tensor, label: tf.Tensor):
    image_bytes = tf.io.read_file(path)
    image = tf.image.decode_image(image_bytes, channels=3, expand_animations=False)
    image.set_shape([None, None, 3])
    image = tf.image.resize(image, IMAGE_SIZE)
    image = tf.cast(image, tf.float32) / 255.0
    return image, label

def build_test_dataset(df: pd.DataFrame) -> tf.data.Dataset:
    paths = df["resolved_image_path"].astype(str).to_numpy()
    labels = df["class_index"].astype(np.int32).to_numpy()
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(decode_resize_image, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

def load_keras_model(path: Path) -> tf.keras.Model:
    return tf.keras.models.load_model(
        path,
        custom_objects={"ResNet50Preprocess": ResNet50Preprocess},
        compile=False,
    )

def classification_report_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    report = classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        output_dict=True,
        zero_division=0,
    )
    return {
        "accuracy": float(report["accuracy"]),
        "macro_f1": float(report["macro avg"]["f1-score"]),
        "weighted_f1": float(report["weighted avg"]["f1-score"]),
    }

def audit_model(spec: dict, df: pd.DataFrame) -> tuple[pd.DataFrame, dict]:
    model_path = Path(spec["model_path"])
    if not model_path.exists():
        return pd.DataFrame(), {
            "model_label": spec["model_label"],
            "experiment_slug": spec["experiment_slug"],
            "model_path": str(model_path),
            "status": "missing_model_artifact",
        }

    print(f"Loading {spec['model_label']} from {model_path}")
    model = load_keras_model(model_path)
    ds = build_test_dataset(df)

    y_true_parts = []
    y_pred_parts = []
    confidence_parts = []

    for images, labels in ds:
        probs = model.predict(images, verbose=0)
        preds = np.argmax(probs, axis=1)
        confidences = np.max(probs, axis=1)

        y_true_parts.append(labels.numpy())
        y_pred_parts.append(preds)
        confidence_parts.append(confidences)

    y_true = np.concatenate(y_true_parts)
    y_pred = np.concatenate(y_pred_parts)
    confidence = np.concatenate(confidence_parts)
    correct = y_true == y_pred
    high_confidence_error = (~correct) & (confidence >= HIGH_CONFIDENCE_THRESHOLD)

    audit_df = df[["image_path", "relative_path", "class_name", "class_index", "source_image_id"]].copy()
    audit_df.insert(0, "experiment_slug", spec["experiment_slug"])
    audit_df.insert(0, "model_label", spec["model_label"])
    audit_df["y_true"] = y_true
    audit_df["y_pred"] = y_pred
    audit_df["predicted_class"] = [class_names[int(idx)] for idx in y_pred]
    audit_df["confidence"] = confidence.astype(float)
    audit_df["correct"] = correct.astype(bool)
    audit_df["high_confidence_error"] = high_confidence_error.astype(bool)

    metrics = classification_report_metrics(y_true, y_pred)
    error_confidences = confidence[~correct]
    summary = {
        "model_label": spec["model_label"],
        "experiment_slug": spec["experiment_slug"],
        "model_path": str(model_path),
        "status": "ok",
        "test_rows": int(len(y_true)),
        "accuracy": metrics["accuracy"],
        "macro_f1": metrics["macro_f1"],
        "weighted_f1": metrics["weighted_f1"],
        "total_errors": int((~correct).sum()),
        "high_confidence_threshold": HIGH_CONFIDENCE_THRESHOLD,
        "high_confidence_errors": int(high_confidence_error.sum()),
        "mean_confidence_on_errors": float(error_confidences.mean()) if len(error_confidences) else 0.0,
    }

    del model
    tf.keras.backend.clear_session()
    gc.collect()
    return audit_df, summary

## Run Audit

This can take time because it loads and evaluates each model sequentially. If GPU memory is limited, restart the kernel and run only a subset of `MODEL_SPECS`.

In [ ]:
all_audit_frames = []
summary_rows = []

for spec in MODEL_SPECS:
    audit_df, summary = audit_model(spec, test_df)
    summary_rows.append(summary)
    if not audit_df.empty:
        all_audit_frames.append(audit_df)
    display(pd.DataFrame([summary]))

summary_df = pd.DataFrame(summary_rows)
audit_predictions_df = pd.concat(all_audit_frames, ignore_index=True) if all_audit_frames else pd.DataFrame()

display(summary_df.sort_values(["status", "accuracy"], ascending=[True, False]))

## Save Outputs

In [ ]:
summary_path = OUTPUT_DIR / "all_cv_model_confidence_audit_summary.csv"
predictions_path = OUTPUT_DIR / "all_cv_model_confidence_audit_predictions.csv"

summary_df.to_csv(summary_path, index=False)
audit_predictions_df.to_csv(predictions_path, index=False)

print("Saved summary:", summary_path)
print("Saved prediction-level audit:", predictions_path)

report_columns = [
    "model_label",
    "accuracy",
    "macro_f1",
    "weighted_f1",
    "total_errors",
    "high_confidence_errors",
    "mean_confidence_on_errors",
    "status",
]
display(summary_df[report_columns].sort_values("accuracy", ascending=False))

## LaTeX Table Helper

After running the audit, use the displayed table below to update the report's CV comparison table.

In [ ]:
latex_df = summary_df[summary_df["status"] == "ok"].copy()
latex_df = latex_df.sort_values("accuracy", ascending=False)
latex_df["accuracy"] = latex_df["accuracy"].map(lambda x: f"{x:.5f}")
latex_df["macro_f1"] = latex_df["macro_f1"].map(lambda x: f"{x:.5f}")
latex_df["weighted_f1"] = latex_df["weighted_f1"].map(lambda x: f"{x:.5f}")
latex_df["high_confidence_errors"] = latex_df["high_confidence_errors"].astype(int)

display(latex_df[["model_label", "accuracy", "macro_f1", "weighted_f1", "high_confidence_errors"]])
print(latex_df[["model_label", "accuracy", "macro_f1", "weighted_f1", "high_confidence_errors"]].to_latex(index=False, escape=False))